# Regression Models

This notebook tests whether last-minute engagement is associated with assessment score. The main explanatory variable is `cramming_ratio`, defined as final 7-day clicks divided by final 28-day clicks.

In [1]:
from pathlib import Path

import pandas as pd
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "analysis_assessment_34874.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.shape

(1658, 22)

## Prepare Variables

The models use `score` as the outcome. The main variable is `cramming_ratio`. Later models add total click volume and student background controls.

In [2]:
model_df = df[
    [
        "score",
        "cramming_ratio",
        "total_clicks_28d",
        "active_learning_days",
        "studied_credits",
        "num_of_prev_attempts",
        "highest_education",
        "gender",
        "age_band",
        "disability",
    ]
].copy()

model_df = model_df.dropna()

model_df.shape

(1658, 10)

## Model 1: Score And Cramming Ratio Only

This is the simplest test of the relationship.

In [3]:
model_1 = smf.ols("score ~ cramming_ratio", data=model_df).fit(cov_type="HC3")
print(model_1.summary())

                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.017
Model:                            OLS   Adj. R-squared:                  0.017
Method:                 Least Squares   F-statistic:                     23.17
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           1.62e-06
Time:                        15:29:20   Log-Likelihood:                -7032.4
No. Observations:                1658   AIC:                         1.407e+04
Df Residuals:                    1656   BIC:                         1.408e+04
Df Model:                           1                                         
Covariance Type:                  HC3                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept         75.5916      0.779     97.

## Model 2: Add Total Click Volume

This tests whether cramming matters beyond the total amount of online engagement.

In [4]:
model_2 = smf.ols(
    "score ~ cramming_ratio + total_clicks_28d",
    data=model_df,
).fit(cov_type="HC3")

print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.119
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     92.29
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           1.00e-38
Time:                        15:29:20   Log-Likelihood:                -6941.9
No. Observations:                1658   AIC:                         1.389e+04
Df Residuals:                    1655   BIC:                         1.391e+04
Df Model:                           2                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           66.8257      1.106  

## Model 3: Add Learning Consistency And Student Controls

This model adds active learning days and student background variables. Categorical variables are wrapped in `C()`.

In [5]:
model_3 = smf.ols(
    "score ~ cramming_ratio + total_clicks_28d + active_learning_days "
    "+ studied_credits + num_of_prev_attempts "
    "+ C(highest_education) + C(gender) + C(age_band) + C(disability)",
    data=model_df,
).fit(cov_type="HC3")

print(model_3.summary())

                            OLS Regression Results                            
Dep. Variable:                  score   R-squared:                       0.185
Model:                            OLS   Adj. R-squared:                  0.178
Method:                 Least Squares   F-statistic:                     29.17
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           3.23e-65
Time:                        15:29:20   Log-Likelihood:                -6877.4
No. Observations:                1658   AIC:                         1.378e+04
Df Residuals:                    1644   BIC:                         1.386e+04
Df Model:                          13                                         
Covariance Type:                  HC3                                         
                                                          coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------

## Compare Main Results

The table below focuses on the `cramming_ratio` coefficient across models.

In [6]:
models = {
    "Model 1": model_1,
    "Model 2": model_2,
    "Model 3": model_3,
}

comparison = []
for name, model in models.items():
    comparison.append(
        {
            "model": name,
            "cramming_ratio_coef": model.params["cramming_ratio"],
            "robust_se": model.bse["cramming_ratio"],
            "p_value": model.pvalues["cramming_ratio"],
            "r_squared": model.rsquared,
            "n": int(model.nobs),
        }
    )

comparison_df = pd.DataFrame(comparison).round(4)
comparison_df

,model,cramming_ratio_coef,robust_se,p_value,r_squared,n
0,Model 1,-10.1671,2.1122,0.0000,0.0173,1658
1,Model 2,-3.3434,2.1570,0.1211,0.1189,1658
2,Model 3,0.3270,2.1244,0.8777,0.1848,1658


## Save Regression Summary

In [7]:
comparison_path = REPORTS_DIR / "regression_model_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)

summary_path = REPORTS_DIR / "regression_results.txt"
with summary_path.open("w") as f:
    f.write("Model 1\n")
    f.write(model_1.summary().as_text())
    f.write("\n\nModel 2\n")
    f.write(model_2.summary().as_text())
    f.write("\n\nModel 3\n")
    f.write(model_3.summary().as_text())

print(f"Saved comparison table to {comparison_path}")
print(f"Saved full regression results to {summary_path}")

Saved comparison table to /Users/sunnyxie/Documents/Codex/2026-07-09/i/oulad-last-minute-learning/reports/regression_model_comparison.csv
Saved full regression results to /Users/sunnyxie/Documents/Codex/2026-07-09/i/oulad-last-minute-learning/reports/regression_results.txt
